In [4]:


import math
import copy

X = "X"
O = "O"
EMPTY = None

# --------------------------------------------
# CẤU HÌNH BOARD
# --------------------------------------------
N = 5          # kích thước bàn cờ NxN (có thể đổi 5, 7, 10...)
WIN_LEN = 4    # số quân liên tiếp để thắng (3 hoặc 4 hoặc 5 tùy đề)

# --------------------------------------------
# KHỞI TẠO BOARD
# --------------------------------------------
def initial_state():
    return [[EMPTY for _ in range(N)] for _ in range(N)]

# --------------------------------------------
# NGƯỜI ĐI HIỆN TẠI
# --------------------------------------------
def player(board):
    count = sum(row.count(X) + row.count(O) for row in board)
    return X if count % 2 == 0 else O

# --------------------------------------------
# DANH SÁCH NƯỚC ĐI
# --------------------------------------------
def actions(board):
    moves = []
    for i in range(N):
        for j in range(N):
            if board[i][j] == EMPTY:
                moves.append((i, j))
    return moves

# --------------------------------------------
# TẠO TRẠNG THÁI MỚI SAU NƯỚC ĐI
# --------------------------------------------
def result(board, action):
    (i, j) = action
    new_board = copy.deepcopy(board)
    new_board[i][j] = player(board)
    return new_board

# --------------------------------------------
# KIỂM TRA THẮNG (liên tiếp WIN_LEN)
# --------------------------------------------
def check_line(line):
    """
    line là danh sách ký tự, ví dụ ['X', 'X', 'O', 'X']
    """
    countX = countO = 0
    for c in line:
        if c == X:
            countX += 1
            countO = 0
            if countX >= WIN_LEN:
                return X
        elif c == O:
            countO += 1
            countX = 0
            if countO >= WIN_LEN:
                return O
        else:
            countX = countO = 0
    return None

def winner(board):
    # kiểm tra hàng
    for i in range(N):
        w = check_line(board[i])
        if w: return w

    # kiểm tra cột
    for j in range(N):
        col = [board[i][j] for i in range(N)]
        w = check_line(col)
        if w: return w

    # kiểm tra chéo chính
    for base in range(N):
        diag1 = []
        diag2 = []
        for k in range(N):
            if 0 <= base + k < N:
                diag1.append(board[k][base + k])
                diag2.append(board[base + k][k])
        if len(diag1) >= WIN_LEN:
            w = check_line(diag1)
            if w: return w
        if len(diag2) >= WIN_LEN:
            w = check_line(diag2)
            if w: return w

    # kiểm tra chéo phụ
    for base in range(N):
        diag3 = []
        diag4 = []
        for k in range(N):
            if 0 <= base - k < N:
                diag3.append(board[k][base - k])
                diag4.append(board[base - k][k])
        if len(diag3) >= WIN_LEN:
            w = check_line(diag3)
            if w: return w
        if len(diag4) >= WIN_LEN:
            w = check_line(diag4)
            if w: return w

    return None

# --------------------------------------------
# KẾT THÚC GAME?
# --------------------------------------------
def terminal(board):
    if winner(board) is not None:
        return True
    for i in range(N):
        for j in range(N):
            if board[i][j] == EMPTY:
                return False
    return True

# --------------------------------------------
# UTILITY (giá trị của trạng thái kết thúc)
# --------------------------------------------
def utility(board):
    w = winner(board)
    if w == X: return 1
    if w == O: return -1
    return 0

# --------------------------------------------
# MINIMAX
# --------------------------------------------
def max_value(board, depth):
    if depth == 0 or terminal(board):
        return utility(board)

    v = -math.inf
    for action in actions(board):
        v = max(v, min_value(result(board, action), depth - 1))
    return v

def min_value(board, depth):
    if depth == 0 or terminal(board):
        return utility(board)

    v = math.inf
    for action in actions(board):
        v = min(v, max_value(result(board, action), depth - 1))
    return v

def minimax(board, depth=3):
    """
    depth: giới hạn độ sâu để giảm thời gian tính toán (rất quan trọng cho NxN)
    depth = 3 hoặc 4 là hợp lý.
    """
    turn = player(board)
    best_move = None

    if turn == X:
        best = -math.inf
        for action in actions(board):
            val = min_value(result(board, action), depth - 1)
            if val > best:
                best = val
                best_move = action
        return best_move

    else:
        best = math.inf
        for action in actions(board):
            val = max_value(result(board, action), depth - 1)
            if val < best:
                best = val
                best_move = action
        return best_move

# --------------------------------------------
# CHẠY THỬ
# --------------------------------------------
if __name__ == "__main__":
    board = initial_state()
    print(f"Bạn chọn X hoặc O (bàn cờ {N}x{N}, thắng {WIN_LEN} liên tiếp): ")
    user = input().upper()

    while True:
        for r in board:
            print(r)

        if terminal(board):
            w = winner(board)
            if w: print("Kết thúc! Người thắng:", w)
            else: print("Hòa!")
            break

        if player(board) == user:
            i = int(input("row = "))
            j = int(input("col = "))
            if board[i][j] == EMPTY:
                board[i][j] = user
        else:
            print("AI đang suy nghĩ (Minimax)…")
            move = minimax(board, depth=3)
            board = result(board, move)

Bạn chọn X hoặc O (bàn cờ 5x5, thắng 4 liên tiếp): 
x
[None, None, None, None, None]
[None, None, None, None, None]
[None, None, None, None, None]
[None, None, None, None, None]
[None, None, None, None, None]
row = 2
col = 2
[None, None, None, None, None]
[None, None, None, None, None]
[None, None, 'X', None, None]
[None, None, None, None, None]
[None, None, None, None, None]
AI đang suy nghĩ (Minimax)…
['O', None, None, None, None]
[None, None, None, None, None]
[None, None, 'X', None, None]
[None, None, None, None, None]
[None, None, None, None, None]
row = 2
col = 1
['O', None, None, None, None]
[None, None, None, None, None]
[None, 'X', 'X', None, None]
[None, None, None, None, None]
[None, None, None, None, None]
AI đang suy nghĩ (Minimax)…
['O', 'O', None, None, None]
[None, None, None, None, None]
[None, 'X', 'X', None, None]
[None, None, None, None, None]
[None, None, None, None, None]
row = 2
col = 3
['O', 'O', None, None, None]
[None, None, None, None, None]
[None, 'X', 'X', 